# 01 — Python for AI Engineers

**Fase:** 1 — Fundamenter  
**Estimert tid:** 2–3 timer  
**Hva du bygger:** Et lite program som henter filmlister fra et falskt API, parser JSON-svar, og skriver rene hjelpefunksjoner — de samme mønstrene du vil se i all AI-kode fremover.

---

## Hvorfor dette notabooks kommer først

AI-kode er bare Python. Men det er noen Python-mønstre du vil se **om og om igjen** i biblioteker som `openai`, `anthropic`, `langchain` og `fastapi`. Denne notatboken dekker akkurat disse mønstrene — ikke Python fra scratch, men Python slik det brukes i produksjons-AI-systemer.

Du trenger ikke ha lest noen Python-bok. Hvis du kan skrive en for-løkke og definere en funksjon, er du klar.

## Forutsetninger

- Ingen tidligere notatbøker kreves
- Grunnleggende Python: variabler, løkker, funksjoner
- Python 3.10+

In [ ]:
# Ingen eksterne biblioteker trengs for denne notatboken
import json
import time
from dataclasses import dataclass, field
from typing import Optional

print("Klar! Python-versjon:")
import sys
print(sys.version)

---

## Del 1: List comprehensions og generator expressions

**Hva det er:** En kortere måte å lage lister på.  
**Hvorfor det er viktig i AI:** Du vil transformere og filtrere store mengder tekst, tokens, og resultater hele tiden.

```python
# Gammel måte (lengre)
result = []
for x in liste:
    if betingelse(x):
        result.append(transform(x))

# List comprehension (kortere og raskere)
result = [transform(x) for x in liste if betingelse(x)]
```

In [ ]:
# Eksempel: du har en liste med meldinger fra en chatbot
meldinger = [
    {"rolle": "bruker", "tekst": "Hva er en embedding?"},
    {"rolle": "assistent", "tekst": "En embedding er en liste med tall..."},
    {"rolle": "bruker", "tekst": "Kan du gi et eksempel?"},
    {"rolle": "assistent", "tekst": "Selvfølgelig! For eksempel..."},
]

# Hent bare brukerens meldinger
brukermeldinger = [m["tekst"] for m in meldinger if m["rolle"] == "bruker"]
print("Brukermeldinger:", brukermeldinger)

# Gjør alle tekster til store bokstaver (dummy-eksempel på transformasjon)
store_bokstaver = [m["tekst"].upper() for m in meldinger]
print("\nFørste melding i caps:", store_bokstaver[0])

In [ ]:
# Generator expression: som list comprehension, men bruker ikke minne for hele lista på en gang
# Viktig når du jobber med tusenvis av dokumenter

tekster = ["hei verden", "  spaces rundt  ", "STORE BOKSTAVER", "normal tekst"]

# Generator (lazy — evaluerer ett element om gangen)
rens_gen = (t.strip().lower() for t in tekster)

# Bruk generatoren i en løkke
for renset in rens_gen:
    print(renset)

---

## Del 2: Dataclasses — Strukturerte dataobjekter

**Hva det er:** En måte å lage klasser med automatisk `__init__`, `__repr__` etc.  
**Hvorfor det er viktig i AI:** Du representerer alltid strukturerte data — meldinger, dokumenter, søkeresultater, agenthandlinger. `dataclass` er ryddigere enn ordbøker og sikrere enn løse klasser.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Dokument:
    """Representerer et dokument i et RAG-system."""
    id: str
    innhold: str
    kilde: str
    relevans_score: Optional[float] = None  # Settes etter søk
    metadata: dict = field(default_factory=dict)  # Unngå mutable default!

# Lag noen dokumenter
dok1 = Dokument(
    id="spk-001",
    innhold="AFP er en tidligpensjonsordning for ansatte i offentlig sektor.",
    kilde="spk.no",
    metadata={"dato": "2024-01-15", "kategori": "pensjon"}
)

dok2 = Dokument(
    id="spk-002",
    innhold="Alderspensjon fra SPK utbetales livsvarig fra 67 år.",
    kilde="spk.no",
    relevans_score=0.92
)

print(dok1)
print()
print(f"Dokument {dok2.id} har relevans: {dok2.relevans_score}")

---

## Del 3: Type hints

**Hva det er:** Merkelapper som forteller hva slags data en funksjon forventer og returnerer.  
**Hvorfor det er viktig i AI:** AI-systemer er komplekse. Type hints gjør koden lesbar, gjør det mulig for editoren å hjelpe deg, og fanger feil tidlig.

```python
# Uten type hints — hva er meldinger? Hva returneres?
def lag_prompt(meldinger, system):
    ...

# Med type hints — krystallklart
def lag_prompt(meldinger: list[dict], system: str) -> str:
    ...
```

In [ ]:
from typing import Optional

def filtrer_dokumenter(
    dokumenter: list[Dokument],
    min_score: float = 0.7,
    maks_antall: Optional[int] = None
) -> list[Dokument]:
    """
    Filtrerer dokumenter basert på relevans-score.
    
    Args:
        dokumenter: Liste med Dokument-objekter
        min_score: Minimum relevans-score (0.0 til 1.0)
        maks_antall: Maksimalt antall dokumenter å returnere
    
    Returns:
        Filtrert og sortert liste med dokumenter
    """
    relevante = [
        d for d in dokumenter
        if d.relevans_score is not None and d.relevans_score >= min_score
    ]
    
    # Sorter etter score (høyest først)
    relevante.sort(key=lambda d: d.relevans_score, reverse=True)
    
    if maks_antall:
        return relevante[:maks_antall]
    return relevante

# Test
alle_dok = [
    Dokument("1", "Tekst om AFP", "spk.no", relevans_score=0.95),
    Dokument("2", "Generell info", "nav.no", relevans_score=0.45),
    Dokument("3", "Alderspensjon", "spk.no", relevans_score=0.82),
    Dokument("4", "Urelatert", "vg.no"),  # Ingen score
]

topp_dok = filtrer_dokumenter(alle_dok, min_score=0.8, maks_antall=2)
for d in topp_dok:
    print(f"Score {d.relevans_score:.2f}: {d.innhold[:40]}...")

---

## Del 4: JSON-parsing

**Hva det er:** JSON er dataformatet alle API-er bruker. Du vil parse JSON fra LLM-svar, API-responser, konfigurasjonsfiler — hele dagen.  
**Viktig:** LLMs returnerer ofte tekst som *ser ut som* JSON, men som trenger litt rensing.

In [ ]:
import json

# Typisk LLM-svar: JSON pakket inn i markdown
llm_svar = """
Her er analysen din:

```json
{
  "sentiment": "positiv",
  "score": 0.87,
  "nøkkelord": ["pensjon", "trygghet", "fremtid"],
  "sammendrag": "Brukeren er fornøyd med pensjonsordningen."
}
```
"""

def ekstraher_json(tekst: str) -> dict:
    """Trekker ut JSON fra en tekst som kan inneholde markdown-kodeblokker."""
    # Finn innhold mellom ```json og ```
    if "```json" in tekst:
        start = tekst.find("```json") + 7
        slutt = tekst.find("```", start)
        json_tekst = tekst[start:slutt].strip()
    elif "```" in tekst:
        start = tekst.find("```") + 3
        slutt = tekst.find("```", start)
        json_tekst = tekst[start:slutt].strip()
    else:
        json_tekst = tekst.strip()
    
    return json.loads(json_tekst)

resultat = ekstraher_json(llm_svar)
print("Sentiment:", resultat["sentiment"])
print("Score:", resultat["score"])
print("Nøkkelord:", resultat["nøkkelord"])

In [ ]:
# Robust JSON-parsing med feilhåndtering
def sikker_json_parse(tekst: str, fallback: dict = None) -> dict:
    """Parser JSON sikkert — returnerer fallback hvis parsing feiler."""
    if fallback is None:
        fallback = {}
    try:
        return ekstraher_json(tekst)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Advarsel: JSON-parsing feilet ({e}). Returnerer fallback.")
        return fallback

# Test med ugyldig JSON
ugyldig = "Dette er ikke JSON!"
resultat = sikker_json_parse(ugyldig, fallback={"feil": True, "melding": "Parsing feilet"})
print(resultat)

---

## Del 5: async/await — Asynkron kode

**Hva det er:** En måte å kjøre kode som *venter* (f.eks. på et API-kall) uten å blokkere hele programmet.  
**Hvorfor det er viktig i AI:** Alle LLM-API-kall tar tid (0.5–10 sekunder). Med `async` kan du sende 10 forespørsler *parallelt* i stedet for én om gangen.

```
SYNKRON (én om gangen):    [---req1---][---req2---][---req3---]  = 9 sekunder
ASYNKRON (parallelt):      [---req1---]
                            [---req2---]                        = 3 sekunder
                            [---req3---]
```

In [ ]:
import asyncio
import time

# Simulerer et LLM-API-kall (ekte kall tar 1-5 sekunder)
async def kall_llm(prompt: str, forsinkelse: float = 1.0) -> str:
    """Simulerer et asynkront API-kall til en LLM."""
    await asyncio.sleep(forsinkelse)  # 'await' = vent uten å blokkere
    return f"Svar på: '{prompt[:30]}...'"

# Asynkron funksjon som kaller LLM én gang
async def ett_kall():
    start = time.time()
    svar = await kall_llm("Hva er AFP?", forsinkelse=1.0)
    print(f"Svar: {svar} ({time.time()-start:.1f}s)")

# Kjør i Jupyter med await direkte
await ett_kall()

In [ ]:
# Parallelle kall med asyncio.gather()
async def mange_kall_parallelt():
    spørsmål = [
        "Hva er AFP?",
        "Hvordan beregnes alderspensjon?",
        "Hva er uførepensjon?",
    ]
    
    start = time.time()
    
    # Alle kall sendes SAMTIDIG
    svar = await asyncio.gather(
        *[kall_llm(q, forsinkelse=1.5) for q in spørsmål]
    )
    
    elapsed = time.time() - start
    print(f"Fikk {len(svar)} svar på {elapsed:.1f}s (i stedet for {1.5*len(svar):.1f}s)")
    for s in svar:
        print(" -", s)

await mange_kall_parallelt()

---

## Del 6: Context managers (`with`-setninger)

**Hva det er:** En måte å håndtere ressurser (filer, tilkoblinger, timere) slik at de alltid lukkes/ryddes opp — selv om koden krasjer.  
**Hvorfor det er viktig i AI:** Du åpner filkobling til databaser, API-klienter, og fil-håndterere hele tiden.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def tidsmaaling(operasjon: str):
    """Context manager som måler tid på en operasjon."""
    print(f"Starter: {operasjon}")
    start = time.time()
    try:
        yield  # Koden inne i 'with'-blokken kjøres her
    finally:
        elapsed = time.time() - start
        print(f"Ferdig: {operasjon} ({elapsed:.3f}s)")

# Bruk context manageren
with tidsmaaling("Laster inn dokumenter"):
    time.sleep(0.2)  # Simulerer lasting

with tidsmaaling("Indekserer vektordatabase"):
    time.sleep(0.3)  # Simulerer indeksering

---

## Miniprosjekt: Meldingsparser for chatbot-logg

Sett sammen alt du har lært. Du får en JSON-logg fra en chatbot og skal:
1. Parse JSON-en
2. Filtrere og transformere meldingene
3. Lage en oppsummeringsrapport som en dataklasse
4. Simulere asynkrone API-kall

In [ ]:
import asyncio
import json
from dataclasses import dataclass, field
from typing import Optional

# --- Data ---
chatbot_logg_json = """
[
  {"id": 1, "rolle": "bruker", "tekst": "Hva er AFP?", "tokens": 5},
  {"id": 2, "rolle": "assistent", "tekst": "AFP er Avtalefestet pensjon...", "tokens": 120},
  {"id": 3, "rolle": "bruker", "tekst": "Hvem kan søke?", "tokens": 4},
  {"id": 4, "rolle": "assistent", "tekst": "Du kan søke hvis du er...", "tokens": 89},
  {"id": 5, "rolle": "bruker", "tekst": "Takk!", "tokens": 2},
  {"id": 6, "rolle": "assistent", "tekst": "Bare hyggelig!", "tokens": 10}
]
"""

@dataclass
class SamtaleStatistikk:
    totalt_meldinger: int
    bruker_meldinger: int
    assistent_meldinger: int
    totalt_tokens: int
    snitt_tokens_assistent: float
    spørsmål: list[str] = field(default_factory=list)

def analyser_samtale(json_tekst: str) -> SamtaleStatistikk:
    """Parser en chatbot-logg og returnerer statistikk."""
    meldinger = json.loads(json_tekst)
    
    bruker_mld = [m for m in meldinger if m["rolle"] == "bruker"]
    assistent_mld = [m for m in meldinger if m["rolle"] == "assistent"]
    
    totalt_tokens = sum(m["tokens"] for m in meldinger)
    snitt_tokens = sum(m["tokens"] for m in assistent_mld) / len(assistent_mld)
    spørsmål = [m["tekst"] for m in bruker_mld if "?" in m["tekst"]]
    
    return SamtaleStatistikk(
        totalt_meldinger=len(meldinger),
        bruker_meldinger=len(bruker_mld),
        assistent_meldinger=len(assistent_mld),
        totalt_tokens=totalt_tokens,
        snitt_tokens_assistent=snitt_tokens,
        spørsmål=spørsmål
    )

stats = analyser_samtale(chatbot_logg_json)
print("=== Samtalestatistikk ===")
print(f"Totalt meldinger: {stats.totalt_meldinger}")
print(f"Bruker: {stats.bruker_meldinger}, Assistent: {stats.assistent_meldinger}")
print(f"Totalt tokens brukt: {stats.totalt_tokens}")
print(f"Snitt tokens per assistent-melding: {stats.snitt_tokens_assistent:.1f}")
print(f"Spørsmål stilt: {stats.spørsmål}")

In [ ]:
# Simuler at vi sender alle brukerspørsmål til en LLM for re-analyse
async def re_analyser_spørsmål(spørsmål: list[str]) -> list[dict]:
    """Sender alle spørsmål til LLM parallelt."""
    
    async def analyser_ett(spm: str) -> dict:
        await asyncio.sleep(0.5)  # Simulerer API-kall
        return {
            "spørsmål": spm,
            "kategori": "pensjon",
            "kompleksitet": "medium"
        }
    
    # Parallelt!
    resultater = await asyncio.gather(*[analyser_ett(s) for s in spørsmål])
    return list(resultater)

with tidsmaaling("Re-analyserer spørsmål"):
    # Jupyter støtter 'await' direkte i celler
    analyser = await re_analyser_spørsmål(stats.spørsmål)

for a in analyser:
    print(f"'{a['spørsmål']}' → kategori: {a['kategori']}, kompleksitet: {a['kompleksitet']}")

---

## Oppsummering

| Konsept | Hva det gjør | Når du bruker det i AI |
|---------|-------------|----------------------|
| List comprehension | Kompakt listebygging | Filtrere/transformere tekster, tokens |
| Generator | Lazy evaluering | Store datasett uten å laste alt i minnet |
| Dataclass | Strukturerte objekter | Meldinger, dokumenter, søkeresultater |
| Type hints | Dokumenterer datatyper | Alle funksjoner i produksjonskode |
| JSON-parsing | Parser API-svar | LLM-output, API-responser |
| async/await | Parallelle API-kall | LLM-kall, embedding-generering |
| Context manager | Ressurshåndtering | DB-tilkoblinger, timing, filhåndtering |

---

## Hva er neste steg?

**Neste notatbok: `02_numpy_pandas.ipynb`**  
Du lærer å jobbe med tall og tabeller i Python — grunnlaget for å forstå embedding-vektorer (som er bare store lister med tall) og håndtere datasett. Numpy og Pandas er biblioteker du vil se i nesten all data-/AI-kode.